<a href="https://colab.research.google.com/github/shilpitha-03/VideoRAG/blob/domain-prep/videorag_run_domain_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive mounted at /content/drive/MyDrive/")

Mounted at /content/drive
✓ Drive mounted at /content/drive/MyDrive/


In [ ]:
import torch
import subprocess

print("=== CUDA state after each import ===")

print(f"Baseline: {torch.cuda.is_initialized()}")

import os
print(f"After os: {torch.cuda.is_initialized()}")

import sys
print(f"After sys: {torch.cuda.is_initialized()}")

# Check nvidia-smi without touching torch
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.used',
                     '--format=csv,noheader'],
                    capture_output=True, text=True)
print(f"GPU: {gpu.stdout.strip()}")
print(f"After nvidia-smi: {torch.cuda.is_initialized()}")

=== CUDA state after each import ===
Baseline: False
After os: False
After sys: False
GPU: NVIDIA A100-SXM4-80GB, 0 MiB
After nvidia-smi: False


Cell 2 — Runtime verificationRun this first. Confirms you got the A100 and high RAM before spending time on anything else.

In [ ]:
import subprocess
import os

# Verify GPU
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu_info.stdout)

# Verify RAM
ram_info = subprocess.run(['free', '-h'], capture_output=True, text=True)
print(ram_info.stdout)

# Verify disk space on local disk
disk_info = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True)
print(disk_info.stdout)

if os.path.exists('/content/drive/MyDrive'):
    print("✓ Drive confirmed mounted")
else:
    print("✗ Drive NOT mounted - run Cell 1 first")

Mon May 18 03:18:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

You want to see: A100 40GB in the GPU output, 80GB+ RAM, and 100GB+ free on /content/. If you see a T4 instead, go to Runtime → Change runtime type and select A100 + High RAM before continuing.

Cell 3 — Define all paths

In [ ]:
import os
### videos will stay on disk
if not os.path.exists('/content/drive/MyDrive'):
    raise RuntimeError("Drive not mounted - run Cell 1 first")
print("\n✓ Drive confirmed mounted, all paths ready.")

# Drive paths - persistent across sessions
drive_root = '/content/drive/MyDrive/videorag_project'
drive_paths = {
    'trimmed_videos': f'{drive_root}/trimmed_videos',
    # 'workdir':        f'{drive_root}/workdir',
    'workdir_3b1b':     f'{drive_root}/workdir_3b1b',       # rename existing
    'workdir_surgical': f'{drive_root}/workdir_surgical',
    'inspection':     f'{drive_root}/inspection_outputs',
}
workdir = drive_paths['workdir_surgical']
# Local disk paths - temporary, recreated each session
local_paths = {
    'weights':    '/content/model_weights',
    'whisper':    '/content/model_weights/faster-distil-whisper-large-v3',
    'minicpm':    '/content/model_weights/MiniCPM-V-2_6-int4',
    'imagebind':  '/content/model_weights/imagebind_huge.pth',
    'raw_videos': '/content/raw_videos',
    'cache':      '/content/videorag_cache',
}

# Create Drive folders
print("=== Drive folders (persistent) ===")
for name, path in drive_paths.items():
    os.makedirs(path, exist_ok=True)
    print(f"✓ {name}: {path}")

# Create local folders (.pth is a file not a folder, skip it)
print("\n=== Local folders (this session only) ===")
for name, path in local_paths.items():
    if not path.endswith('.pth'):
        os.makedirs(path, exist_ok=True)
        print(f"✓ {name}: {path}")

print("\nAll paths ready.")


✓ Drive confirmed mounted, all paths ready.
=== Drive folders (persistent) ===
✓ trimmed_videos: /content/drive/MyDrive/videorag_project/trimmed_videos
✓ workdir_3b1b: /content/drive/MyDrive/videorag_project/workdir_3b1b
✓ workdir_surgical: /content/drive/MyDrive/videorag_project/workdir_surgical
✓ inspection: /content/drive/MyDrive/videorag_project/inspection_outputs

=== Local folders (this session only) ===
✓ weights: /content/model_weights
✓ whisper: /content/model_weights/faster-distil-whisper-large-v3
✓ minicpm: /content/model_weights/MiniCPM-V-2_6-int4
✓ raw_videos: /content/raw_videos
✓ cache: /content/videorag_cache

All paths ready.


In [ ]:
import os, shutil
old = f'{drive_root}/workdir'
new = f'{drive_root}/workdir_3b1b'
if os.path.exists(old) and not os.path.exists(new):
    shutil.move(old, new)
    print(f"✓ Renamed {old} → {new}")
os.makedirs(f'{drive_root}/workdir_surgical', exist_ok=True)

Every cell from here references drive_paths or local_paths. One place to change if anything moves.

Cell 4 — Clone repo and verify branch

In [ ]:
import os
###branch to checkout is
# Clone your fork - only runs if not already cloned
if not os.path.exists('/content/VideoRAG'):
    !git clone https://github.com/shilpitha-03/VideoRAG.git /content/VideoRAG

# Change working directory permanently for this session
%cd /content/VideoRAG/VideoRAG-algorithm

# Checkout the branch with the HuggingFace embedding fix
!git checkout domain-prep

# Show current branch and recent commits
!git branch
!git log --oneline -3

# Actively verify the code change is present
# If this prints ✗, the wrong branch is checked out
!grep -n "huggingface.co" videorag/_llm.py \
    && echo "✓ HuggingFace endpoint confirmed in _llm.py" \
    || echo "✗ Change not found - wrong branch or push failed"

Cloning into '/content/VideoRAG'...
remote: Enumerating objects: 645, done.
remote: Counting objects: 100% (369/369), done.
remote: Compressing objects: 100% (198/198), done.
remote: Total 645 (delta 233), reused 194 (delta 168), pack-reused 276 (from 2)
Receiving objects: 100% (645/645), 6.71 MiB | 22.61 MiB/s, done.
Resolving deltas: 100% (318/318), done.
/content/VideoRAG/VideoRAG-algorithm
Branch 'domain-prep' set up to track remote branch 'domain-prep' from 'origin'.
Switched to a new branch 'domain-prep'
* domain-prep
  main
85bf104 (HEAD -> domain-prep, origin/domain-prep) changes to caption.py, prompt.py, asr.py
2463ea6 branch: starting surgical domain adaptation
b392892 (origin/hf-embedding-fix) loading whisper from cpu
491:#             "https://api-inference.huggingface.co/models/BAAI/bge-m3",
✓ HuggingFace endpoint confirmed in _llm.py


In [ ]:
%cd /content/VideoRAG/VideoRAG-algorithm
!git pull origin domain-prep
!git log --oneline -3

/content/VideoRAG/VideoRAG-algorithm
From https://github.com/shilpitha-03/VideoRAG
 * branch            domain-prep -> FETCH_HEAD
Already up to date.
85bf104 (HEAD -> domain-prep, origin/domain-prep) changes to caption.py, prompt.py, asr.py
2463ea6 branch: starting surgical domain adaptation
b392892 (origin/hf-embedding-fix) loading whisper from cpu


Cell 5a — Install dependencies

In [ ]:
# Install PyTorch - no version pin, resolves for this Colab environment
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

import torch
print(f"PyTorch: {torch.__version__}")
# Do NOT call torch.cuda.is_available() or any cuda function here
# Any torch.cuda call initializes CUDA which breaks multiprocessing fork in Cell 10
print("✓ PyTorch installed - CUDA will initialize lazily when first needed by a model")

PyTorch: 2.10.0+cu128
✓ PyTorch installed - CUDA will initialize lazily when first needed by a model


Cell 5b — Rest of dependencies

In [ ]:
!pip install accelerate -q
!pip install bitsandbytes -q
!pip install moviepy==1.0.3 -q
!pip install timm ftfy regex einops fvcore eva-decord==0.6.1 iopath -q
!pip install ctranslate2==4.4.0 faster_whisper==1.0.3 -q
!pip install hnswlib xxhash nano-vectordb neo4j -q
# !pip install transformers -q
!pip install transformers==4.43.3 -q
!pip install tiktoken openai tenacity -q
!pip install yt-dlp -q
!pip install ollama==0.5.3 -q

!pip install --no-deps \
    git+https://github.com/facebookresearch/pytorchvideo.git@28fe037d212663c6a24f373b94cc5d478c8c1a1d -q
!pip install --no-deps \
    git+https://github.com/facebookresearch/ImageBind.git@3fcf5c9039de97f6ff5528ee4a9dce903c5979b3 -q

print("✓ All dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 129.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.7/34.7 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 132.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 34.7 MB/s e

In [ ]:
import importlib
import transformers
importlib.reload(transformers)
print(transformers.__version__)

4.43.3


5c-patch the broken file in place

In [ ]:
# Patch pytorchvideo's broken import of functional_tensor
# This module was removed in torchvision 0.17 (torch 2.2+)
# pytorchvideo hasn't been updated to handle this
# The patch creates a compatibility shim in place of the missing module

import torchvision.transforms.functional as F
import sys
import types

# Create a fake functional_tensor module with the functions
# pytorchvideo actually uses from it
fake_module = types.ModuleType('torchvision.transforms.functional_tensor')

# These are the specific functions pytorchvideo imports from functional_tensor
# They still exist in torchvision.transforms.functional under the same names
funcs_to_copy = [
    'rgb_to_grayscale',
    'adjust_brightness',
    'adjust_contrast',
    'adjust_saturation',
    'adjust_hue',
]

for func_name in funcs_to_copy:
    if hasattr(F, func_name):
        setattr(fake_module, func_name, getattr(F, func_name))

# Register the fake module so pytorchvideo's import finds it
sys.modules['torchvision.transforms.functional_tensor'] = fake_module

print("✓ pytorchvideo compatibility patch applied")

# Verify the patch works by importing what was failing
try:
    from pytorchvideo.transforms import augmix
    print("✓ pytorchvideo imports successfully after patch")
except Exception as e:
    print(f"✗ Patch incomplete: {e}")

✓ pytorchvideo compatibility patch applied
✓ pytorchvideo imports successfully after patch


 Cell 5d Create a symlink that makes cuDNN 8 available where the system expects it.

In [ ]:
import subprocess
import os

# ctranslate2 bundles cuDNN 8 internally
# bitsandbytes looks for libcudnn_ops_infer.so.8 in system paths
# Solution: symlink ctranslate2's bundled cuDNN 8 to where bitsandbytes looks

cudnn8_source = '/usr/local/lib/python3.12/dist-packages/ctranslate2.libs/libcudnn-463fd6d5.so.8.9.7'

# Create symlinks for all the specific .so.8 names that get requested
symlink_targets = [
    '/usr/lib/x86_64-linux-gnu/libcudnn_ops_infer.so.8',
    '/usr/lib/x86_64-linux-gnu/libcudnn_cnn_infer.so.8',
    '/usr/lib/x86_64-linux-gnu/libcudnn_cnn_train.so.8',
    '/usr/lib/x86_64-linux-gnu/libcudnn_ops_train.so.8',
    '/usr/lib/x86_64-linux-gnu/libcudnn_adv_infer.so.8',
    '/usr/lib/x86_64-linux-gnu/libcudnn_adv_train.so.8',
]

for target in symlink_targets:
    if not os.path.exists(target):
        result = subprocess.run(
            ['ln', '-sf', cudnn8_source, target],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"✓ Created: {os.path.basename(target)}")
        else:
            print(f"✗ Failed: {target} — {result.stderr}")
    else:
        print(f"✓ Already exists: {os.path.basename(target)}")

# Update the dynamic linker cache
subprocess.run(['ldconfig'], capture_output=True)
print("\n✓ Library cache updated")
print("cuDNN 8 ops now available to bitsandbytes and other libraries")

✓ Created: libcudnn_ops_infer.so.8
✓ Created: libcudnn_cnn_infer.so.8
✓ Created: libcudnn_cnn_train.so.8
✓ Created: libcudnn_ops_train.so.8
✓ Created: libcudnn_adv_infer.so.8
✓ Created: libcudnn_adv_train.so.8

✓ Library cache updated
cuDNN 8 ops now available to bitsandbytes and other libraries


In [ ]:
import torch
print(f"CUDA initialized: {torch.cuda.is_initialized()}")

CUDA initialized: False


Cell 6 — Download model weights to local disk

In [ ]:
!pip install -q huggingface_hub

In [ ]:
import os
import subprocess

# ── WHISPER ───────────────────────────────────────────────────────────────
# Check for actual weight file, not just directory existence
# 4.0K directory = empty = git lfs didn't download weights
whisper_model_bin = f"{local_paths['whisper']}/model.bin"
whisper_ok = os.path.exists(whisper_model_bin) and os.path.getsize(whisper_model_bin) > 1e9

if not whisper_ok:
    print("Downloading Whisper (~1.5GB)...")
    if os.path.exists(local_paths['whisper']):
        import shutil
        shutil.rmtree(local_paths['whisper'])
    !git lfs install
    !git clone https://huggingface.co/Systran/faster-distil-whisper-large-v3 \
        {local_paths['whisper']}
else:
    print("✓ Whisper already on local disk with real weights")

# Verify
size_gb = os.path.getsize(whisper_model_bin) / 1e9
print(f"  model.bin: {size_gb:.2f} GB {'✓' if size_gb > 1 else '✗ pointer file!'}")

# # ── MINICPM-V ─────────────────────────────────────────────────────────────
# # MiniCPM-V-2_6-int4 splits weights across two shard files
# # minicpm_shard = f"{local_paths['minicpm']}/model-00001-of-00002.bin"
# minicpm_shard1 = f"{local_paths['minicpm']}/model-00001-of-00008.safetensors"
# minicpm_shard2 = f"{local_paths['minicpm']}/model-00008-of-00008.safetensors"
# minicpm_ok = (os.path.exists(minicpm_shard1) and
#               os.path.getsize(minicpm_shard1) > 1e9 and
#               os.path.exists(minicpm_shard2) and
#               os.path.getsize(minicpm_shard2) > 1e9)

# if not minicpm_ok:
#     print("\nDownloading MiniCPM-V full model (~16GB, one-time)...")
#     if os.path.exists(local_paths['minicpm']):
#         import shutil
#         shutil.rmtree(local_paths['minicpm'])
#     !git lfs install
#     !git clone https://huggingface.co/openbmb/MiniCPM-V-2_6 \
#         {local_paths['minicpm']}
# else:
#     print("\n✓ MiniCPM-V already on local disk with real weights")

# # Verify both shards
# # Check first shard exists and is real
# for shard in ['model-00001-of-00008.safetensors']:
#     shard_path = f"{local_paths['minicpm']}/{shard}"
#     if os.path.exists(shard_path):
#         size_gb = os.path.getsize(shard_path) / 1e9
#         print(f"  {shard}: {size_gb:.2f} GB "
#               f"{'✓' if size_gb > 1 else '✗ pointer file!'}")
#     else:
#         print(f"  {shard}: ✗ NOT FOUND")

# ── MINICPM-V ─────────────────────────────────────────────────────────────

from huggingface_hub import login, snapshot_download
import os
import shutil

# LOGIN FIRST
# Paste your HF token when prompted
login()

# Expected shard files
minicpm_shard1 = f"{local_paths['minicpm']}/model-00001-of-00004.safetensors"
minicpm_shard4 = f"{local_paths['minicpm']}/model-00004-of-00004.safetensors"
# previously assumed 8 shards, but its actually 4 shards only.
# Verify existing download
minicpm_ok = (
    os.path.exists(minicpm_shard1) and
    os.path.getsize(minicpm_shard1) > 1e9 and
    os.path.exists(minicpm_shard4) and
    os.path.getsize(minicpm_shard4) > 1e9
)

if not minicpm_ok:

    print("\nDownloading MiniCPM-V full model (~16GB, one-time)...")

    # Remove broken partial download
    if os.path.exists(local_paths['minicpm']):
        shutil.rmtree(local_paths['minicpm'])

    # Download model
    snapshot_download(
        repo_id="openbmb/MiniCPM-V-2_6",
        local_dir=local_paths['minicpm'],
        local_dir_use_symlinks=False,
        resume_download=True
    )

else:
    print("\n✓ MiniCPM-V already on local disk with real weights")

# Verify shards
for shard in [
    'model-00001-of-00004.safetensors',
    'model-00004-of-00004.safetensors'
]:
    shard_path = f"{local_paths['minicpm']}/{shard}"

    if os.path.exists(shard_path):
        size_gb = os.path.getsize(shard_path) / 1e9

        print(
            f"  {shard}: {size_gb:.2f} GB "
            f"{'✓' if size_gb > 1 else '✗ pointer/small file!'}"
        )
    else:
        print(f"  {shard}: ✗ NOT FOUND")

# ── IMAGEBIND ─────────────────────────────────────────────────────────────
# ImageBind is a single .pth file downloaded via wget - reliable
imagebind_ok = os.path.exists(local_paths['imagebind']) and \
               os.path.getsize(local_paths['imagebind']) > 1e9

if not imagebind_ok:
    print("\nDownloading ImageBind (~2GB)...")
    !wget -q https://dl.fbaipublicfiles.com/imagebind/imagebind_huge.pth \
        -O {local_paths['imagebind']}
else:
    print("\n✓ ImageBind already on local disk")

size_gb = os.path.getsize(local_paths['imagebind']) / 1e9
print(f"  imagebind_huge.pth: {size_gb:.2f} GB {'✓' if size_gb > 1 else '✗ too small!'}")

# ── FINAL SUMMARY ─────────────────────────────────────────────────────────
print("\n=== Model verification summary ===")
checks = [
    ("Whisper",    whisper_model_bin,  1e9),
    ("ImageBind",  local_paths['imagebind'], 1e9),
    ("MiniCPM shard1", minicpm_shard1, 1e9),
]
all_ok = True
for name, path, min_size in checks:
    ok = os.path.exists(path) and os.path.getsize(path) > min_size
    print(f"  {'✓' if ok else '✗'} {name}")
    if not ok:
        all_ok = False

if all_ok:
    print("\n✓ All models ready. Safe to continue.")
else:
    raise RuntimeError("✗ Some models missing or incomplete - check output above")

✓ Whisper already on local disk with real weights
  model.bin: 1.51 GB ✓



✓ MiniCPM-V already on local disk with real weights
  model-00001-of-00004.safetensors: 4.87 GB ✓
  model-00004-of-00004.safetensors: 2.06 GB ✓

✓ ImageBind already on local disk
  imagebind_huge.pth: 4.80 GB ✓

=== Model verification summary ===
  ✓ Whisper
  ✓ ImageBind
  ✓ MiniCPM shard1

✓ All models ready. Safe to continue.


In [ ]:
import torch
print(f"CUDA initialized: {torch.cuda.is_initialized()}")

CUDA initialized: False


Cell 7 — Symlinks only, no clone, correct branch

In [ ]:
import os

# Symlink models from local disk into paths the code expects
# The VideoRAG code has hardcoded relative paths for these models
# Symlinks make local disk files appear where the code looks

links = [
    (local_paths['whisper'],
     '/content/VideoRAG/VideoRAG-algorithm/faster-distil-whisper-large-v3'),
    (local_paths['minicpm'],
     '/content/VideoRAG/VideoRAG-algorithm/MiniCPM-V-2_6-int4'),
]

for src, dst in links:
    if not os.path.exists(dst):
        os.symlink(src, dst)
        print(f"✓ Symlink created: {dst}")
    else:
        print(f"✓ Symlink already exists: {dst}")

# ImageBind expects a .checkpoints folder specifically
os.makedirs('/content/VideoRAG/VideoRAG-algorithm/.checkpoints', exist_ok=True)
imagebind_link = '/content/VideoRAG/VideoRAG-algorithm/.checkpoints/imagebind_huge.pth'
if not os.path.exists(imagebind_link):
    os.symlink(local_paths['imagebind'], imagebind_link)
    print(f"✓ Symlink created: {imagebind_link}")
else:
    print(f"✓ Symlink already exists: {imagebind_link}")

# Verify all symlinks resolve to actual files on local disk
# os.path.exists follows the symlink and checks the target exists
print("\n=== Symlink verification ===")
for name, path in [("Whisper",   links[0][1]),
                    ("MiniCPM-V", links[1][1]),
                    ("ImageBind", imagebind_link)]:
    exists = os.path.exists(path)
    print(f"{'✓' if exists else '✗'} {name}: {'accessible' if exists else 'BROKEN - target not found'}")

✓ Symlink already exists: /content/VideoRAG/VideoRAG-algorithm/faster-distil-whisper-large-v3
✓ Symlink already exists: /content/VideoRAG/VideoRAG-algorithm/MiniCPM-V-2_6-int4
✓ Symlink already exists: /content/VideoRAG/VideoRAG-algorithm/.checkpoints/imagebind_huge.pth

=== Symlink verification ===
✓ Whisper: accessible
✓ MiniCPM-V: accessible
✓ ImageBind: accessible


Cell 8 — Download and trim videos directly to Drive

The raw download goes to local disk, gets trimmed immediately, only the trimmed version touches Drive. Raw file is deleted right after trimming. Clean.

Cell 9 — Set API key and configure VideoRAG

In [ ]:
import os
import zipfile

# Extract ZIP from Drive
zip_path = "/content/drive/MyDrive/videorag_project/jan_15.zip"
extract_dir = "/content/uploads"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
print("✓ Extracted")

# Find final_video.mp4 inside the extracted folder
video_path = None
for root, _, files in os.walk(extract_dir):
    for f in files:
        if f == "final_video.mp4":
            video_path = os.path.join(root, f)
            break

if video_path is None:
    raise FileNotFoundError("final_video.mp4 not found in ZIP")

size_mb = os.path.getsize(video_path) / 1e6
print(f"✓ {video_path}: {size_mb:.1f} MB")

# This is what insert_video() receives
video_paths = [video_path]

✓ Extracted
✓ /content/uploads/final_video.mp4: 520.8 MB


In [ ]:
import nest_asyncio
nest_asyncio.apply()
from google.colab import userdata

import os
import sys
import httpx
import asyncio
import types
import torch

sys.path.insert(0, '/content/VideoRAG/VideoRAG-algorithm')

# ── CRITICAL: verify CUDA is not initialized ──────────────────────────────
# MiniCPM-V captioning runs in a forked subprocess
# If CUDA is initialized in parent, fork fails with:
# "Cannot re-initialize CUDA in forked subprocess"
if torch.cuda.is_initialized():
    raise RuntimeError(
        "CUDA already initialized - restart runtime and do NOT call any "
        "torch.cuda functions before this cell. Check Cell 5a and remove "
        "any cuda calls."
    )
print("✓ CUDA not initialized - safe for multiprocessing fork")

# ── pytorchvideo patch ────────────────────────────────────────────────────
import torchvision.transforms.functional as F
fake_module = types.ModuleType('torchvision.transforms.functional_tensor')
for func_name in ['rgb_to_grayscale','adjust_brightness','adjust_contrast',
                  'adjust_saturation','adjust_hue']:
    if hasattr(F, func_name):
        setattr(fake_module, func_name, getattr(F, func_name))
sys.modules['torchvision.transforms.functional_tensor'] = fake_module
print("✓ pytorchvideo patch applied")

# ── clear cached modules ──────────────────────────────────────────────────
mods_to_remove = [key for key in sys.modules if 'videorag' in key]
for mod in mods_to_remove:
    del sys.modules[mod]

# ── API keys ──────────────────────────────────────────────────────────────
# os.environ["DEEPSEEK_API_KEY"] = "your-deepseek-key-here"
os.environ["DEEPSEEK_API_KEY"] = userdata.get('DS_TOKEN')

print("\n=== API Key Format Check ===")
key = os.environ.get("DEEPSEEK_API_KEY", "")
if key.startswith("sk-"):
    print(f"✓ DEEPSEEK_API_KEY: {key[:6]}...{key[-4:]}")
else:
    raise ValueError("✗ DEEPSEEK_API_KEY looks wrong")

# ── Test DeepSeek ─────────────────────────────────────────────────────────
print("\n=== Testing DeepSeek API ===")
async def test_deepseek():
    async with httpx.AsyncClient() as client:
        response = await client.post(
            "https://api.deepseek.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {os.environ['DEEPSEEK_API_KEY']}",
                     "Content-Type": "application/json"},
            json={"model": "deepseek-chat",
                  "messages": [{"role": "user", "content": "Reply with one word: working"}],
                  "max_tokens": 10},
            timeout=30.0
        )
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]

result = await test_deepseek()
print(f"✓ DeepSeek response: {result}")

# ── Import config ─────────────────────────────────────────────────────────
# DO NOT call get_bge_model() here - initializes CUDA
# bge-m3 loads lazily after all forked processes complete
from videorag._llm import deepseek_bge_config
from videorag import VideoRAG, QueryParam
print("\n✓ deepseek_bge_config imported")
print("✓ bge-m3 will load lazily after captioning subprocess completes")

# ── Set paths ─────────────────────────────────────────────────────────────
workdir = drive_paths['workdir_surgical']

# video_paths is already set by the extraction cell
# Just verify it's defined and the files exist
print("\n=== Input Videos ===")
for vp in video_paths:
    exists = os.path.exists(vp)
    size_mb = os.path.getsize(vp) / 1e6 if exists else 0
    print(f"{'✓' if exists else '✗'} {os.path.basename(vp)}: {size_mb:.0f}MB")
    if not exists:
        raise FileNotFoundError(f"Missing: {vp}")

print(f"\nWorkdir: {workdir}")
print("✓ All checks passed. Safe to run indexing.")

✓ CUDA not initialized - safe for multiprocessing fork
✓ pytorchvideo patch applied

=== API Key Format Check ===
✓ DEEPSEEK_API_KEY: sk-758...21f6

=== Testing DeepSeek API ===
✓ DeepSeek response: working

✓ deepseek_bge_config imported
✓ bge-m3 will load lazily after captioning subprocess completes

=== Input Videos ===
✓ final_video.mp4: 521MB

Workdir: /content/drive/MyDrive/videorag_project/workdir_surgical
✓ All checks passed. Safe to run indexing.


Cell 9.5 CUDA guard

In [ ]:
import torch
import gc

gc.collect()

# Final gate before indexing
# If CUDA is initialized here, the captioning subprocess will crash
if torch.cuda.is_initialized():
    print("⚠ WARNING: CUDA initialized - captioning fork will fail")
    print("  Restart runtime and check that no torch.cuda calls happen before Cell 10")
else:
    print("✓ CUDA clean - safe to run Cell 10")
    print("  bge-m3 will initialize CUDA only after all subprocess stages complete")

✓ CUDA clean - safe to run Cell 10
  bge-m3 will initialize CUDA only after all subprocess stages complete


Cell 10 — Run indexing

Cell 11 — Inspect workdir structure
This is your first look at what was actually produced.

In [ ]:
import os, json, shutil

# Check what's actually on disk
config_path = f"{local_paths['minicpm']}/config.json"
needs_redownload = True
if os.path.exists(config_path):
    with open(config_path) as f:
        config = json.load(f)
    quant = config.get('quantization_config', None)
    if quant is None:
        print("✓ Full precision model already present, no redownload needed")
        needs_redownload = False
    else:
        print(f"✗ Quantized model present ({quant.get('quant_method', '?')}), need full precision")
else:
    print("✗ No config.json found, need download")

# if needs_redownload:
#     if os.path.exists(local_paths['minicpm']):
#         print("Deleting old model...")
#         shutil.rmtree(local_paths['minicpm'])
#     print("Downloading full precision MiniCPM-V-2_6 (~16GB, 15-20 min)...")
#     !git lfs install -q
#     !git clone https://huggingface.co/openbmb/MiniCPM-V-2_6 {local_paths['minicpm']}

#     # Verify it's actually full precision now
#     with open(f"{local_paths['minicpm']}/config.json") as f:
#         config = json.load(f)
#     assert config.get('quantization_config') is None, "Still quantized after download — something went wrong"
#     print("✓ Full precision confirmed")

✓ Full precision model already present, no redownload needed


In [ ]:
import time

start_time = time.time()

print("=== Initializing VideoRAG ===")
print(f"Config: deepseek_bge_config")
print(f"  LLM: deepseek-chat (entity extraction, filtering, generation)")
print(f"  Embeddings: BAAI/bge-m3 local on A100 (dim=1024)")
print(f"Workdir: {workdir}\n")

videorag = VideoRAG(
    llm=deepseek_bge_config,
    working_dir=workdir
)

print("=== Starting Indexing Pipeline ===")
print("Stage 1: Split videos into 30s clips")
print("Stage 2: ASR - Whisper transcribes each clip")
print("Stage 3: VLM - MiniCPM-V captions each clip")
print("Stage 4: LLM - DeepSeek extracts entities + relationships per chunk")
print("Stage 5: LLM - DeepSeek merges and synthesizes entity descriptions")
print("Stage 6: Embeddings - bge-m3 embeds chunks and entities on GPU")
print("Stage 7: ImageBind embeds each clip visually")
print("\nWatch the output below for stage transitions...\n")

videorag.insert_video(video_path_list=video_paths)

elapsed = time.time() - start_time
print(f"\n✓ Indexing complete in {elapsed/60:.1f} minutes")
print(f"Index saved to: {workdir}")

=== Initializing VideoRAG ===
Config: deepseek_bge_config
  LLM: deepseek-chat (entity extraction, filtering, generation)
  Embeddings: BAAI/bge-m3 local on A100 (dim=1024)
Workdir: /content/drive/MyDrive/videorag_project/workdir_surgical

=== Starting Indexing Pipeline ===
Stage 1: Split videos into 30s clips
Stage 2: ASR - Whisper transcribes each clip
Stage 3: VLM - MiniCPM-V captions each clip
Stage 4: LLM - DeepSeek extracts entities + relationships per chunk
Stage 5: LLM - DeepSeek merges and synthesizes entity descriptions
Stage 6: Embeddings - bge-m3 embeds chunks and entities on GPU
Stage 7: ImageBind embeds each clip visually

Watch the output below for stage transitions...



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Spliting Video final_video: 100%|██████████| 23/23 [00:07<00:00,  3.13it/s]


Found annotations CSV: /content/drive/MyDrive/videorag_project/workdir_surgical/../annotations/final_video.csv
✓ Mapped 11 annotations onto 23 clips for final_video


Captioning Video final_video:   0%|          | 0/23 [00:00<?, ?it/s]WARNING:py.warnings:/usr/local/lib/python3.12/dist-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(

Encoding Video Segments final_video: 100%|██████████| 12/12 [01:52<00:00,  9.37s/it]


...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

✓ bge-m3 loaded


✓ Indexing complete in 14.9 minutes
Index saved to: /content/drive/MyDrive/videorag_project/workdir_surgical


Setup before inspection - switch to cpu now, everything in index is saved to drive already

In [ ]:
# === Setup for inspection (CPU runtime, no GPU needed) ===
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json

# Paths (same as indexing session)
drive_root = '/content/drive/MyDrive/videorag_project'
drive_paths = {
    'trimmed_videos': f'{drive_root}/trimmed_videos',
    'workdir':        f'{drive_root}/workdir_surgical',
    'inspection':     f'{drive_root}/inspection_outputs_surgical',
}
workdir = drive_paths['workdir']

os.makedirs(drive_paths['inspection'], exist_ok=True)

# Verify workdir has content
files = os.listdir(workdir)
print(f"Workdir: {workdir}")
print(f"Files: {len(files)}")
for f in sorted(files):
    size_kb = os.path.getsize(os.path.join(workdir, f)) / 1e3
    print(f"  {f} ({size_kb:.1f} KB)")

print(f"\n✓ Ready for inspection — no GPU, no repo clone, no models needed")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Workdir: /content/drive/MyDrive/videorag_project/workdir_surgical
Files: 11
  _cache (4.1 KB)
  final_video_captions.json (28.9 KB)
  final_video_transcripts.json (2.8 KB)
  graph_chunk_entity_relation.graphml (61.2 KB)
  kv_store_llm_response_cache.json (42.9 KB)
  kv_store_text_chunks.json (33.4 KB)
  kv_store_video_path.json (0.1 KB)
  kv_store_video_segments.json (38.8 KB)
  vdb_chunks.json (38.7 KB)
  vdb_entities.json (366.5 KB)
  vdb_video_segment_feature.json (127.5 KB)

✓ Ready for inspection — no GPU, no repo clone, no models needed


In [ ]:
import os
import json

print("=== WORKDIR STRUCTURE ===\n")

# Check workdir actually has content
contents = list(os.walk(workdir))
if len(contents) <= 1 and not os.listdir(workdir):
    raise RuntimeError("Workdir is empty - indexing may not have completed. Check Cell 10 output.")

for root, dirs, files in os.walk(workdir):
    dirs[:] = [d for d in dirs if d != '_cache']
    level = root.replace(workdir, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        filepath = os.path.join(root, file)
        size_kb = os.path.getsize(filepath) / 1e3
        print(f"{indent}  {file} ({size_kb:.1f} KB)")

cache_dir = os.path.join(workdir, '_cache')
if os.path.exists(cache_dir):
    mp3_files = []
    mp4_files = []
    for root, dirs, files in os.walk(cache_dir):
        for f in files:
            if f.endswith('.mp3'): mp3_files.append(f)
            if f.endswith('.mp4'): mp4_files.append(f)
    print(f"\n_cache/ (shown separately)")
    print(f"  Audio clips (.mp3): {len(mp3_files)}")
    print(f"  Video clips (.mp4): {len(mp4_files)}")
    print(f"  Total clips: {len(mp3_files)} (each is one 30-second segment)")

=== WORKDIR STRUCTURE ===

workdir_surgical/
  final_video_transcripts.json (2.8 KB)
  final_video_captions.json (28.9 KB)
  vdb_video_segment_feature.json (127.5 KB)
  kv_store_video_segments.json (38.8 KB)
  kv_store_video_path.json (0.1 KB)
  kv_store_llm_response_cache.json (42.9 KB)
  kv_store_text_chunks.json (33.4 KB)
  vdb_entities.json (366.5 KB)
  vdb_chunks.json (38.7 KB)
  graph_chunk_entity_relation.graphml (61.2 KB)

_cache/ (shown separately)
  Audio clips (.mp3): 0
  Video clips (.mp4): 0
  Total clips: 0 (each is one 30-second segment)


Cell 12 — Read and display transcripts

In [ ]:
import json
import os

print("=== TRANSCRIPTS AND CAPTIONS PER CLIP ===\n")
print("Raw output of ASR (Whisper) + VLM (MiniCPM-V)")
print("For each 30-second clip: what was heard + what was seen\n")

chunk_files = []
for root, dirs, files in os.walk(workdir):
    for f in files:
        if 'chunk' in f.lower() or 'caption' in f.lower() or 'text' in f.lower():
            chunk_files.append(os.path.join(root, f))

if not chunk_files:
    print("No chunk/caption files found yet.")
    print("These files are written during indexing.")
    print("Run this cell after Cell 10 completes.")
else:
    all_data = {}
    for cf in chunk_files[:3]:
        print(f"\n--- File: {os.path.basename(cf)} ---")
        with open(cf) as fp:
            data = json.load(fp)
        all_data[os.path.basename(cf)] = data

        if isinstance(data, dict):
            print(f"Type: dict with {len(data)} entries")
            for i, (key, value) in enumerate(data.items()):
                if i >= 2: break
                print(f"\nEntry key: {key}")
                print(json.dumps(value, indent=2)[:800])
                print("...")
        elif isinstance(data, list):
            print(f"Type: list with {len(data)} entries")
            for item in data[:2]:
                print(json.dumps(item, indent=2)[:800])
                print("...")

    # Save actual data, not a placeholder
    inspection_file = f"{drive_paths['inspection']}/transcripts_and_captions.json"
    with open(inspection_file, 'w') as f:
        json.dump(all_data, f, indent=2)
    print(f"\n✓ Actual data saved to Drive: {inspection_file}")

=== TRANSCRIPTS AND CAPTIONS PER CLIP ===

Raw output of ASR (Whisper) + VLM (MiniCPM-V)
For each 30-second clip: what was heard + what was seen


--- File: final_video_captions.json ---
Type: dict with 23 entries

Entry key: 0
"The video depicts a medical procedure, likely within the nasal cavity. Initially, it shows a close-up view of what appears to be a nasal passage with visible mucus and some debris or foreign material, possibly indicating an examination for congestion or infection. The intensity is set at 45%, suggesting the use of magnification tools. As the video progresses, the camera zooms in closer on the nasal lining, highlighting its texture and color variations, which range from pinkish tones to areas with redness, potentially indicating inflammation.Subsequently, the perspective shifts slightly, showing a more defined curvature of the nasal passage and the presence of moisture, indicating either natural nasal fluid or post-anesthetic effects if local anesthesia was admi

Cell 13 — Read and display raw entity/relationship extraction per chunk

In [ ]:
import json
import os
import xml.etree.ElementTree as ET

print("=== ENTITIES AND RELATIONSHIPS ===\n")

# ── Part 1: Entity embeddings (vdb_entities.json) ──
vdb_path = os.path.join(workdir, 'vdb_entities.json')
with open(vdb_path) as f:
    vdb = json.load(f)

entities = vdb.get('data', [])
print(f"Total entities: {len(entities)}")
print(f"Embedding dim: {vdb.get('embedding_dim')}\n")

print("Sample entities (first 10):")
for ent in entities[:10]:
    print(f"  {ent.get('entity_name', '?')}")

# ── Part 2: Knowledge graph (GraphML) ──
graph_path = os.path.join(workdir, 'graph_chunk_entity_relation.graphml')
tree = ET.parse(graph_path)
root = tree.getroot()

# GraphML uses a namespace
ns = {'g': 'http://graphml.graphstruct.org/graphml'}
# Try auto-detect namespace
for elem in root.iter():
    if '}' in elem.tag:
        ns['g'] = elem.tag.split('}')[0].strip('{')
        break

nodes = root.findall('.//g:node', ns) or root.findall('.//{http://graphml.graphstruct.org/graphml}node') or root.findall('.//node')
edges = root.findall('.//g:edge', ns) or root.findall('.//{http://graphml.graphstruct.org/graphml}edge') or root.findall('.//edge')

print(f"\n=== KNOWLEDGE GRAPH ===")
print(f"Nodes (entities): {len(nodes)}")
print(f"Edges (relationships): {len(edges)}\n")

# Show first 5 nodes with their data
print("Sample nodes:")
for node in nodes[:5]:
    node_id = node.get('id', '?')
    # Extract data fields from <data> child elements
    fields = {d.get('key', '?'): (d.text or '')[:150] for d in node}
    print(f"\n  Node: {node_id}")
    for k, v in fields.items():
        print(f"    {k}: {v}")

print("\nSample edges:")
for edge in edges[:5]:
    src = edge.get('source', '?')
    tgt = edge.get('target', '?')
    fields = {d.get('key', '?'): (d.text or '')[:150] for d in edge}
    print(f"\n  {src} → {tgt}")
    for k, v in fields.items():
        print(f"    {k}: {v}")

=== ENTITIES AND RELATIONSHIPS ===

Total entities: 66
Embedding dim: 1024

Sample entities (first 10):
  "NASAL CAVITY"
  "ENDOSCOPE"
  "NASAL ACCESS AND PREP"
  "LOCAL ANESTHESIA"
  "DECONGESTION"
  "SURGICAL INSTRUMENT"
  "PATIENT"
  "MEDICAL PROFESSIONAL"
  "NASAL PASSAGE"
  "MUCUS"

=== KNOWLEDGE GRAPH ===
Nodes (entities): 69
Edges (relationships): 85

Sample nodes:

  Node: "NASAL CAVITY"
    d0: "GEO"
    d1: "The interior space of the nose where the ethmoidectomy is performed, containing turbinates, blood vessels, and mucosal tissues."<SEP>"The internal an
    d2: chunk-ee04f772f18fbf88e33e31dd61333c18<SEP>chunk-c339dee97b50877a48399e06aaafa32d<SEP>chunk-60f0bd4853404e27149dd16dec9b1ac8<SEP>chunk-7ab29ea846bffc0

  Node: "ENDOSCOPE"
    d0: "ORGANIZATION"
    d1: "The endoscope is the medical device used to capture the magnified, illuminated views of the nasal cavity during the examination and procedure, as ind
    d2: chunk-46d44e73c6b1749d72e7af4324585338

  Node: "NASAL ACC

Cell 14 — Visualize the merged knowledge graph

In [ ]:
import networkx as nx
import json

# Load the real graph file
graph_path = f"{workdir}/graph_chunk_entity_relation.graphml"
G = nx.read_graphml(graph_path)

print(f"Nodes (entities): {G.number_of_nodes()}")
print(f"Edges (relationships): {G.number_of_edges()}")

# Show all entities with descriptions
print(f"\n{'='*60}")
print("ENTITIES")
print('='*60)
for node_id, attrs in G.nodes(data=True):
    desc = attrs.get('description', '')[:300]
    source = attrs.get('source_id', '')[:200]
    print(f"\n  {node_id}")
    print(f"  Description: {desc}")
    print(f"  Source: {source}")

# Show all relationships
print(f"\n{'='*60}")
print("RELATIONSHIPS")
print('='*60)
for src, tgt, attrs in G.edges(data=True):
    desc = attrs.get('description', '')[:200]
    print(f"\n  {src}  →  {tgt}")
    print(f"  {desc}")

# Find cross-video entities (merged across videos)
print(f"\n{'='*60}")
print("CROSS-VIDEO ENTITIES (appeared in multiple videos)")
print('='*60)
for node_id, attrs in G.nodes(data=True):
    source = attrs.get('source_id', '')
    videos = set()
    for part in source.split('<SEP>'):
        # Extract video name from chunk IDs
        for token in part.split(','):
            token = token.strip()
            if token.startswith('3b1b_'):
                vid = '_'.join(token.split('_')[:5])  # e.g. 3b1b_nn_1_neurons
                videos.add(vid)
    if len(videos) > 1:
        print(f"\n  {node_id} — appears in {len(videos)} videos:")
        for v in sorted(videos):
            print(f"    • {v}")

# Save the real graph data to Drive
graph_data = {
    "stats": {
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges()
    },
    "entities": {n: dict(a) for n, a in G.nodes(data=True)},
    "relationships": [
        {"source": s, "target": t, **dict(a)}
        for s, t, a in G.edges(data=True)
    ]
}
with open(f"{drive_paths['inspection']}/merged_graph_real.json", 'w') as f:
    json.dump(graph_data, f, indent=2)
print(f"\n✓ Real graph data saved to Drive: merged_graph_real.json")

Nodes (entities): 69
Edges (relationships): 85

ENTITIES

  "NASAL CAVITY"
  Description: "The interior space of the nose where the ethmoidectomy is performed, containing turbinates, blood vessels, and mucosal tissues."<SEP>"The internal anatomical space within the nose where the surgical procedures are performed, characterized by red tissue lining, blood vessels, and inflammation."<SEP>
  Source: chunk-ee04f772f18fbf88e33e31dd61333c18<SEP>chunk-c339dee97b50877a48399e06aaafa32d<SEP>chunk-60f0bd4853404e27149dd16dec9b1ac8<SEP>chunk-7ab29ea846bffc0e2e805ece603e37ca<SEP>chunk-46d44e73c6b1749d72e7af

  "ENDOSCOPE"
  Description: "The endoscope is the medical device used to capture the magnified, illuminated views of the nasal cavity during the examination and procedure, as indicated by the light source and close-up perspectives."
  Source: chunk-46d44e73c6b1749d72e7af4324585338

  "NASAL ACCESS AND PREP"
  Description: "Nasal_access_and_prep is the procedural phase identified in the transcr

Cell 15 — Run a retrieval query

In [ ]:
import os
import sys
import multiprocessing

sys.path.insert(0, '/content/VideoRAG/VideoRAG-algorithm')

# ── API KEYS ──────────────────────────────────────────────────────────────
# Must be set again - environment variables don't persist across sessions
os.environ["DEEPSEEK_API_KEY"] = userdata.get("DS_TOKEN")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from videorag._llm import deepseek_bge_config
from videorag import VideoRAG, QueryParam

query = "what is the surgery happening in the videos?"

print(f"Query: {query}\n")
print("Retrieval pipeline:")
print("  Path 1: entity matching via bge-m3 → graph provenance → clips")
print("  Path 2: visual scene description → ImageBind → clips")
print("  Path 3: direct chunk retrieval via bge-m3")
print("  Then: intersect Path1+2 → DeepSeek filter → re-caption → generate\n")

# ── LOAD SAVED INDEX ──────────────────────────────────────────────────────
# This does NOT re-index. It loads the graph and embeddings from workdir.
# Workdir is on Drive so it persists from the indexing session.
multiprocessing.set_start_method('spawn', force=True)

workdir = drive_paths['workdir']

videorag_query = VideoRAG(
    llm=deepseek_bge_config,
    working_dir=workdir
)

# Load MiniCPM-V for query-time re-captioning
# This is the only local model needed at retrieval time
videorag_query.load_caption_model(debug=False)

param = QueryParam(mode="videorag")
param.wo_reference = False  # Include video name + timestamps in response

response = videorag_query.query(query=query, param=param)

print("=== RESPONSE ===\n")
print(response)

# Save to Drive
with open(f"{drive_paths['inspection']}/query_response.txt", 'w') as f:
    f.write(f"Query: {query}\n\n")
    f.write(f"Response:\n{response}")
print(f"\n✓ Response saved to Drive: {drive_paths['inspection']}/query_response.txt")

Query: what is the surgery happening in the videos?

Retrieval pipeline:
  Path 1: entity matching via bge-m3 → graph provenance → clips
  Path 2: visual scene description → ImageBind → clips
  Path 3: direct chunk retrieval via bge-m3
  Then: intersect Path1+2 → DeepSeek filter → re-caption → generate



Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The surgery happening in the videos.
Retrieved Text Segments {'final_video_10', 'final_video_22', 'final_video_11', 'final_video_9', 'final_video_12'}
A surgery is happening in the videos.
Retrieved Visual Segments {'final_video_3', 'final_video_8', 'final_video_0', 'final_video_2'}
9 Video Segments remain after filtering
Remain segments ['final_video_0', 'final_video_2', 'final_video_3', 'final_video_8', 'final_video_9', 'final_video_10', 'final_video_11', 'final_video_12', 'final_video_22']
Keywords: surgery, video


Captioning Segments for Given Query: 100%|██████████| 9/9 [03:06<00:00, 20.78s/it]


=== RESPONSE ===

Based on the retrieved information, the surgery depicted in the video is an **endoscopic ethmoidectomy**, a type of sinus surgery. This procedure is performed to address chronic sinusitis or other nasal blockages by removing portions of the ethmoid bone and associated structures within the nasal cavity.

The surgery is meticulously documented in several phases, beginning with preparation and moving through a series of targeted removals. Here is a breakdown of the key stages observed:

### 1. Nasal Access and Preparation
The initial phase involves preparing the nasal cavity for surgery. This includes a preliminary examination of the nasopharynx to assess its condition [final_video, 0:0:0, 0:0:30]. Following this, **local anesthesia** is administered to numb the area [final_video, 0:0:0, 0:0:30], and a **decongestion** step is performed to reduce swelling and improve visibility for the surgeon [final_video, 0:1:0, 0:1:30].

### 2. Middle Turbinate Management
A key early

What you now have after all 15 cells
On Drive, in inspection_outputs/:

transcripts_and_captions.json — what Whisper heard and what MiniCPM-V saw, per clip
entity_extraction.json — what GPT-4o-mini extracted per chunk, before merging
merged_graph.json — the final unified graph across all 4 videos
query_response.txt — retrieval output with video timestamps

In workdir/:

The complete hybrid index ready for future retrieval sessions without re-running any models

The inspection cells are written to show you the data at each stage. Once you run indexing and see the actual file names and structures, Cells 12-14 may need minor adjustments — the exact JSON keys depend on what nano-graphrag names things internally. When you run them, tell me what they print and I'll refine the inspection code immediately.
Get the API key set up and run Cell 1 first. Tell me what GPU and RAM it shows.

Cell 16 - other queries to test the framework

In [ ]:
queries = [
    # Cross-phase: needs to connect instruments/actions from different surgical phases
    "What instruments are used across different phases of the surgery "
    "and how does their usage change as the procedure progresses?",

    # Specific retrieval: answer should come from one or two clips
    "What anatomical structures are identified or exposed during the "
    "earliest phase of the procedure?",
]

import time

for i, query in enumerate(queries, 1):
    print(f"\n{'='*70}")
    print(f"QUERY {i}")
    print(f"{'='*70}")
    print(f"{query}\n")

    start = time.time()
    param = QueryParam(mode="videorag")
    param.wo_reference = False

    response = videorag_query.query(query=query, param=param)

    elapsed = time.time() - start
    print(f"\n--- Response ({elapsed:.0f}s) ---\n")
    print(response)
    print(f"\n{'='*70}\n")

    with open(f"{drive_paths['inspection']}/query_{i}_response.txt", 'w') as f:
        f.write(f"Query: {query}\n\nResponse:\n{response}")

print("✓ All responses saved to Drive")


QUERY 1
What instruments are used across different phases of the surgery and how does their usage change as the procedure progresses?

The instruments used across different phases of the surgery and how their usage changes as the procedure progresses.
Retrieved Text Segments {'final_video_15', 'final_video_10', 'final_video_16', 'final_video_13', 'final_video_11', 'final_video_14', 'final_video_9', 'final_video_12'}
The usage of different instruments changes across the phases of a surgery as the procedure progresses.
Retrieved Visual Segments {'final_video_3', 'final_video_0', 'final_video_4', 'final_video_2'}
12 Video Segments remain after filtering
Remain segments ['final_video_0', 'final_video_2', 'final_video_3', 'final_video_4', 'final_video_9', 'final_video_10', 'final_video_11', 'final_video_12', 'final_video_13', 'final_video_14', 'final_video_15', 'final_video_16']
Keywords: instruments, surgery, phases, usage change, procedure progresses


Captioning Segments for Given Query: 100%|██████████| 12/12 [04:19<00:00, 21.64s/it]



--- Response (291s) ---

Based on the retrieved video analysis, the endoscopic nasal surgery progresses through several distinct phases, each characterized by the use of specific instruments and techniques. The footage reveals a clear evolution in instrument usage, starting with observational tools and moving to increasingly precise cutting and grasping implements as the surgery advances.

### Phase 1: Nasal Access and Preparation

The initial phase of the surgery focuses on examination and preparation of the nasal cavity. During this stage, the primary instruments observed are a **speculum** or similar retractor and a **probe** or **catheter**. The speculum is used to gently open and stabilize the nasal passage, providing access for the endoscope and other tools [1]. A probe or catheter is then introduced to manipulate tissues and prepare the site for further intervention, such as the administration of local anesthesia [2].

### Phase 2: Decongestion and Tissue Management

As the pro

Captioning Segments for Given Query: 100%|██████████| 9/9 [03:05<00:00, 20.65s/it]



--- Response (211s) ---

Based on the retrieved information, the earliest phase of the procedure is the **Nasal_access_and_prep** phase, which includes the steps of **Initial_examination** and **Local_Anesthesia**.

During this initial phase, the primary anatomical structures identified and exposed are the general interior features of the nasal cavity. The footage focuses on the **nasal mucosa** and the **nasal passageway** itself. The descriptions note the presence of **mucosal structures** [final_video, 0:0:0, 0:0:30], the **nasal septum**, and the **nasopharyngeal walls** [final_video, 0:0:30, 0:1:0]. The visual emphasis is on assessing the condition of the tissue, noting signs of **irritation**, **inflammation**, and **congestion** [final_video, 0:0:0, 0:0:30] [final_video, 0:0:30, 0:1:0].

The procedure at this stage is focused on examination and preparation, with the introduction of a medical instrument (likely a catheter or probe) for the administration of **local anesthesia** 

 The structure is a list, not a tuple or dict. video_frames + [query] is Python list concatenation: [PIL_image_1, PIL_image_2, PIL_image_3, PIL_image_4, PIL_image_5, "the transcript+prompt string"]. Six elements: five PIL Images followed by one string. This list becomes the content of a single chat message wearing the user role.

the chat message format is what the .chat() method expects. Specifically:pythonmsgs = [{'role': 'user', 'content': [<PIL.Image>, <PIL.Image>, ..., <str>]}]This is a multimodal message format (similar to OpenAI's vision API). The model's .chat() method is built to handle a content field that's a list of mixed types — PIL Images and strings interleaved in whatever order you want. You could write [str, Image, str, Image, str] and it would still work; the model preserves order.